In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd
import glob
import os
import re
from difflib import SequenceMatcher

# --- SET YOUR PATH HERE ---
FOLDER_PATH = '/content/drive/MyDrive/week8/'
print("Step 3.1: Loading Notes and identifying version pairs...")

notes_files = glob.glob(FOLDER_PATH + 'notes-*.tsv')
df_notes_list = [pd.read_csv(f, sep='\t', low_memory=False) for f in notes_files]
df_notes = pd.concat(df_notes_list, ignore_index=True)

# Filter for collaborative notes only
valid_flags = [1, 1.0, True]
df_collab = df_notes[df_notes['isCollaborativeNote'].isin(valid_flags)].copy()

# Sort by tweet and time to find chronological updates
df_collab = df_collab.sort_values(by=['tweetId', 'createdAtMillis'])

# Use shift within each tweet group to align v_old with v_new
df_collab['v_old_noteId'] = df_collab.groupby('tweetId')['noteId'].shift(1)
df_collab['v_old_summary'] = df_collab.groupby('tweetId')['summary'].shift(1)
df_collab['v_old_classification'] = df_collab.groupby('tweetId')['classification'].shift(1)

# Drop versions that don't have a predecessor (the very first version of a note)
df_pairs = df_collab.dropna(subset=['v_old_noteId']).copy()

# Rename columns for clarity
df_pairs = df_pairs.rename(columns={
    'noteId': 'v_new_noteId',
    'summary': 'v_new_summary',
    'classification': 'v_new_classification'
})

print(f"Found {len(df_pairs)} consecutive version updates (v_old -> v_new).")


Step 3.1: Loading Notes and identifying version pairs...
Found 6502 consecutive version updates (v_old -> v_new).


In [7]:
print("\n--- Step 3.2: Scanning ALL Rating Files for Suggestions ---")
rating_files = sorted(glob.glob(FOLDER_PATH + 'ratings-*.tsv'))
cols_to_use = ['noteId', 'suggestion']
all_suggestions_list = []

for r_file in rating_files:
    file_name = os.path.basename(r_file)
    print(f"Scanning {file_name}...")

    try:
        # We use chunks to prevent Colab from crashing on large files
        for chunk in pd.read_csv(r_file, sep='\t',
                         usecols=cols_to_use,
                         chunksize=300000,
                         dtype={'noteId': str, 'suggestion': str, 'raterParticipantId': str},
                         low_memory=False):
            # Strip whitespace from column names just in case
            chunk.columns = chunk.columns.str.strip()

            # Filter for rows that actually have a written suggestion
            valid_chunk = chunk[chunk['suggestion'].notna() & (chunk['suggestion'].astype(str).str.strip() != '')]

            if not valid_chunk.empty:
                all_suggestions_list.append(valid_chunk)
    except Exception as e:
        print(f"Skipping {file_name} due to error: {e}")

# Safety check: Only concatenate if we actually found something
if not all_suggestions_list:
    print(" No suggestions found in ANY rating files. Check your data source!")
else:
    df_suggestions = pd.concat(all_suggestions_list, ignore_index=True)
    print(f"Found {len(df_suggestions)} suggestions across all files.")


--- Step 3.2: Scanning ALL Rating Files for Suggestions ---
Scanning ratings-00006.tsv...
Scanning ratings-00007.tsv...
Found 5745 suggestions across all files.


In [15]:
import pandas as pd
import glob

print("Reloading Notes files with forced string formatting for IDs...")

# 1. LOAD NOTES - Force IDs to strings to prevent scientific notation/precision loss
notes_files = sorted(glob.glob(FOLDER_PATH + 'notes-*.tsv'))
df_notes_list = [
    pd.read_csv(f, sep='\t', low_memory=False, dtype={'noteId': str, 'tweetId': str})
    for f in notes_files
]
df_notes = pd.concat(df_notes_list, ignore_index=True)

# 2. FILTER & SORT - Identify Collaborative Notes and sort chronologically
df_collab = df_notes[df_notes['isCollaborativeNote'].isin([1, 1.0, True])].copy()
df_collab = df_collab.sort_values(by=['tweetId', 'createdAtMillis'])

# 3. CREATE VERSION PAIRS - Use shift to find the previous version for each tweet
df_collab['v_old_noteId'] = df_collab.groupby('tweetId')['noteId'].shift(1)
df_collab['v_old_summary'] = df_collab.groupby('tweetId')['summary'].shift(1)
df_collab['v_old_classification'] = df_collab.groupby('tweetId')['classification'].shift(1)

# Drop rows that don't have a previous version (the first version of a note)
df_pairs = df_collab.dropna(subset=['v_old_noteId']).copy()

# Rename current version columns for clarity
df_pairs = df_pairs.rename(columns={
    'noteId': 'v_new_noteId',
    'summary': 'v_new_summary',
    'classification': 'v_new_classification'
})

print(f"Reload complete! Found {len(df_pairs)} version pairs with locked ID precision.")

# 4. PREPARE FOR MERGE - Ensure IDs are clean, trimmed strings
df_pairs['v_old_noteId'] = df_pairs['v_old_noteId'].astype(str).str.strip()
df_suggestions['noteId'] = df_suggestions['noteId'].astype(str).str.strip()

# 5. EXECUTE MERGE - Connect suggestions to the version pairs
df_step3_final = pd.merge(
    df_pairs,
    df_suggestions,
    left_on='v_old_noteId',
    right_on='noteId',
    how='inner'
).drop(columns=['noteId'])

print(f"Matched {len(df_step3_final)} (Version, Suggestion) pairs.")

# Display results if matches were found
if len(df_step3_final) > 0:
    display(df_step3_final[['suggestion', 'v_old_summary', 'v_new_summary']].head())

Reloading Notes files with forced string formatting for IDs...
Reload complete! Found 6502 version pairs with locked ID precision.
Matched 4645 (Version, Suggestion) pairs.


,suggestion,v_old_summary,v_new_summary
0,please elaborate more on &quot;appears to be a...,The post &quot;Test 2&quot; contains no factua...,The statement '1+1 actually equals to 3' is fa...
1,"good note, good job 👍",The statement '1+1 actually equals to 3' is fa...,"In standard mathematics, 1+1=2, not 3. &quot;1..."
2,please explain more details around &quot;metap...,"In standard mathematics, 1+1=2, not 3. &quot;1...",The post simply says &quot;Test 2&quot; and ma...
3,"Note is needed, post is obviously misleading","In standard arithmetic, 1+1=2, not 3. However,...","In standard mathematics, 1+1=2, not 3. The phr..."
4,"This should be a misleading post, context needed","In standard mathematics, 1+1=2, not 3. The phr...","In standard mathematics, 1+1=2, not 3. &quot;1..."


In [16]:
import re
from difflib import SequenceMatcher

def get_text_diff_ratio(s1, s2):
    return SequenceMatcher(None, str(s1), str(s2)).ratio()

def extract_urls(text):
    if pd.isna(text): return set()
    return set(re.findall(r'https?://\S+', str(text)))

print("Auditing changes between version pairs...")

# 1. Character-level diff (Similarity ratio: 1.0 means no change)
df_step3_final['text_similarity'] = df_step3_final.apply(
    lambda x: get_text_diff_ratio(x['v_old_summary'], x['v_new_summary']), axis=1
)

# 2. URL Changes
df_step3_final['v_old_urls'] = df_step3_final['v_old_summary'].apply(extract_urls)
df_step3_final['v_new_urls'] = df_step3_final['v_new_summary'].apply(extract_urls)

df_step3_final['urls_added'] = df_step3_final.apply(lambda x: list(x['v_new_urls'] - x['v_old_urls']), axis=1)
df_step3_final['urls_removed'] = df_step3_final.apply(lambda x: list(x['v_old_urls'] - x['v_new_urls']), axis=1)

# 3. Classification Change (e.g., Misleading -> Not Misleading)
df_step3_final['class_changed'] = df_step3_final['v_old_classification'] != df_step3_final['v_new_classification']

print(f"Audit complete for {len(df_step3_final)} pairs.")
# Statistics for your report
print(f"Average Text Similarity: {df_step3_final['text_similarity'].mean():.2f}")
print(f"Pairs with Classification Changes: {df_step3_final['class_changed'].sum()}")

Auditing changes between version pairs...
Audit complete for 4645 pairs.
Average Text Similarity: 0.50
Pairs with Classification Changes: 982


In [17]:
# 1. Sample 100 pairs
df_sample = df_step3_final.sample(n=100, random_state=42).copy()

# 2. Create the LLM Prompt based on the specified structure
def generate_llm_prompt(row):
    prompt = f"""
    Evaluate whether the following suggestion was incorporated into the updated version of the note.

    ORIGINAL NOTE VERSION:
    {row['v_old_summary']}

    USER SUGGESTION FOR IMPROVEMENT:
    {row['suggestion']}

    NEW NOTE VERSION:
    {row['v_new_summary']}

    TASK:
    Based on the suggestion provided for the original note, did the author incorporate it into the new version?
    Please label the decision as one of the following:
    - ACCEPTED: The suggestion was fully or mostly implemented.
    - PARTIALLY_ACCEPTED: Some elements were implemented, or the change is minor.
    - REJECTED: The suggestion was ignored or the change is unrelated.

    Provide your response in this format:
    LABEL: [Your Label]
    EXPLANATION: [1-2 sentence explanation of your reasoning]
    """
    return prompt.strip()

df_sample['llm_prompt'] = df_sample.apply(generate_llm_prompt, axis=1)

# 3. Save to CSV so you can copy-paste into ChatGPT or use an API
sample_output_path = FOLDER_PATH + 'step3_100_llm_audit_sample.csv'
df_sample.to_csv(sample_output_path, index=False)

print(f"Sample of 100 pairs saved to: {sample_output_path}")

Sample of 100 pairs saved to: /content/drive/MyDrive/week8/step3_100_llm_audit_sample.csv
